# LeetCode 1834 — Single-Threaded CPU bằng Python

Link đề: https://leetcode.com/problems/single-threaded-cpu/ · Độ khó: **Medium** · Chủ đề: Array, Sorting, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def getOrder(self, tasks: List[List[int]]) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. **Mô phỏng sự kiện theo thời gian** (event simulation): biến "thời gian hiện tại" `t`, tập tác vụ **sẵn sàng**, và quy tắc chọn của CPU.
2. Dùng **min-heap với khóa tuple** `(processingTime, index)` để chọn "ngắn nhất, hòa thì chỉ số nhỏ nhất" trong `O(log n)`.
3. Kết hợp **sắp xếp theo thời điểm đến** + con trỏ để đưa tác vụ vào heap đúng lúc; xử lý đúng trường hợp CPU **rảnh** (nhảy thời gian tới tác vụ kế tiếp).
4. So sánh với mô phỏng ngây thơ `O(n²)` và với cách **hai heap** (không sắp xếp trước).
5. Kiểm thử bằng bộ sinh tác vụ ngẫu nhiên có nhiều **hòa** (cùng thời điểm đến, cùng thời gian xử lý).

## 2. Đề bài & ví dụ

Có `n` tác vụ đánh số `0..n−1`, `tasks[i] = [enqueueTime_i, processingTime_i]`: tác vụ `i` **sẵn sàng** từ thời điểm `enqueueTime_i` và cần `processingTime_i` đơn vị thời gian để xử lý.

CPU đơn luồng xử lý tối đa một tác vụ mỗi lúc và hoạt động như sau:
- Nếu CPU rảnh và **không** có tác vụ sẵn sàng, CPU tiếp tục rảnh.
- Nếu CPU rảnh và **có** tác vụ sẵn sàng, CPU chọn tác vụ có **thời gian xử lý ngắn nhất**; nếu hòa, chọn tác vụ có **chỉ số nhỏ nhất**.
- Đã bắt đầu thì xử lý **hết**, không ngắt.
- Xong một tác vụ có thể bắt đầu tác vụ mới **ngay lập tức**.

Trả về thứ tự (dãy chỉ số) các tác vụ được xử lý.

**Ví dụ 1:** `tasks = [[1,2],[2,4],[3,2],[4,1]]` → `[0,2,3,1]`.
- `t=1`: tác vụ 0 sẵn sàng, CPU làm 0 (xong lúc 3).
- `t=3`: sẵn sàng {1 (4), 2 (2)} ⇒ làm 2 (xong lúc 5).
- `t=5`: sẵn sàng {1 (4), 3 (1)} ⇒ làm 3 (xong lúc 6).
- `t=6`: làm 1 (xong lúc 10).

**Ví dụ 2:** `tasks = [[7,10],[7,12],[7,5],[7,4],[7,2]]` → `[4,3,2,0,1]` (tất cả đến lúc 7, làm theo thời gian xử lý tăng dần).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`; `1 ≤ enqueueTime_i, processingTime_i ≤ 10⁹`.

In [1]:
import random
import time
import heapq
from typing import List

examples = [([[1, 2], [2, 4], [3, 2], [4, 1]], [0, 2, 3, 1]), ([[7, 10], [7, 12], [7, 5], [7, 4], [7, 2]], [4, 3, 2, 0, 1])]
for t, exp in examples:
    print(f"tasks={t} -> kỳ vọng {exp}")

tasks=[[1, 2], [2, 4], [3, 2], [4, 1]] -> kỳ vọng [0, 2, 3, 1]
tasks=[[7, 10], [7, 12], [7, 5], [7, 4], [7, 2]] -> kỳ vọng [4, 3, 2, 0, 1]


## 3. Kiến thức nền

### 3.1 So sánh tuple trong Python

Tuple so sánh **từ điển**: `(2, 5) < (2, 7) < (3, 0)`. Vì vậy khóa `(processingTime, index)` tự động thể hiện "ngắn nhất trước, hòa thì chỉ số nhỏ trước".

In [2]:
cands = [(4, 1), (2, 2), (2, 0), (1, 3)]
print(sorted(cands), "| nhỏ nhất:", min(cands))

[(1, 3), (2, 0), (2, 2), (4, 1)] | nhỏ nhất: (1, 3)


### 3.2 `heapq` — min-heap

`heappush(h, x)` / `heappop(h)` `O(log n)`; `h[0]` là phần tử nhỏ nhất. Với tuple, heap so sánh theo thứ tự từ điển như trên.

In [3]:
h = []
for item in [(4, 1), (2, 2), (2, 0), (1, 3)]:
    heapq.heappush(h, item)
print("lấy lần lượt:", [heapq.heappop(h) for _ in range(4)])

lấy lần lượt: [(1, 3), (2, 0), (2, 2), (4, 1)]


### 3.3 Sắp chỉ số theo khóa

`sorted(range(n), key=lambda i: tasks[i][0])` cho thứ tự chỉ số theo thời điểm đến — không làm thay đổi `tasks` (bài nộp không nên phá dữ liệu đầu vào nếu không cần).

In [4]:
tasks = [[3, 2], [1, 2], [2, 4]]
print(sorted(range(len(tasks)), key=lambda i: tasks[i][0]))

[1, 2, 0]


### 3.4 Mô phỏng "nhảy thời gian"

Không cần tăng `t` từng đơn vị (thời gian tới `10⁹`!). Chỉ có hai loại sự kiện: **CPU xong việc** (`t += processingTime`) và **CPU rảnh chờ tác vụ** (`t = thời điểm đến kế tiếp`).

## 4. Ý tưởng nền tảng / trực giác

Mô phỏng tay **Ví dụ 1** với bảng trạng thái:

| t (bắt đầu chọn) | sẵn sàng (chỉ số: thời gian xử lý) | CPU chọn | xong lúc |
|---|---|---|---|
| 1 | {0: 2} | 0 | 3 |
| 3 | {1: 4, 2: 2} | 2 | 5 |
| 5 | {1: 4, 3: 1} | 3 | 6 |
| 6 | {1: 4} | 1 | 10 |

Hai thao tác lặp đi lặp lại: (1) **thêm** các tác vụ có `enqueueTime ≤ t` vào tập sẵn sàng; (2) **lấy** tác vụ "nhỏ nhất" theo `(processingTime, index)`. Thao tác (2) là đặc trưng của **heap**; thao tác (1) dễ nếu tác vụ được **sắp theo thời điểm đến**.

Trường hợp đặc biệt: tập sẵn sàng **rỗng** nhưng còn tác vụ chưa đến ⇒ nhảy `t` tới thời điểm đến sớm nhất còn lại.

In [5]:
tasks = [[5, 1], [1, 1]]          # tác vụ 1 đến trước, CPU rảnh lúc 2 -> nhảy tới 5
order = sorted(range(2), key=lambda i: tasks[i][0])
print("thứ tự đến:", order, "| sau khi xong tác vụ 1 lúc t=2, không còn ai sẵn sàng -> nhảy t tới", tasks[0][0])

thứ tự đến: [1, 0] | sau khi xong tác vụ 1 lúc t=2, không còn ai sẵn sàng -> nhảy t tới 5


## 5. Cách 1 — Brute Force / baseline: mô phỏng, mỗi bước quét mọi tác vụ còn lại

In [6]:
def order_bruteforce(tasks):
    n = len(tasks); done = [False] * n; t = 0; res = []
    for _ in range(n):
        avail = [i for i in range(n) if not done[i] and tasks[i][0] <= t]
        if not avail:                                          # CPU rảnh: nhảy tới tác vụ đến sớm nhất
            t = min(tasks[i][0] for i in range(n) if not done[i])
            avail = [i for i in range(n) if not done[i] and tasks[i][0] <= t]
        j = min(avail, key=lambda i: (tasks[i][1], i))
        done[j] = True; res.append(j); t += tasks[j][1]
    return res

for t, exp in examples:
    got = order_bruteforce(t); print(got, "kỳ vọng", exp); assert got == exp

[0, 2, 3, 1] kỳ vọng [0, 2, 3, 1]
[4, 3, 2, 0, 1] kỳ vọng [4, 3, 2, 0, 1]


**Độ phức tạp:** mỗi lần chọn quét `O(n)` ⇒ `O(n²)` = `10¹⁰` với `n = 10⁵` ⇒ **TLE**.

**Nút thắt:** (1) mỗi bước lọc lại mọi tác vụ để tìm cái sẵn sàng; (2) tìm min trên tập sẵn sàng bằng quét tuyến tính. Cả hai đều cập nhật được dần dần.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (sắp theo thời điểm đến):** sắp chỉ số theo `enqueueTime`; con trỏ `p` chỉ tới tác vụ đầu tiên **chưa** vào tập sẵn sàng. Khi thời gian tăng tới `t`, đẩy mọi tác vụ `p` có `enqueueTime ≤ t` vào tập — mỗi tác vụ được đẩy **đúng một lần**.

**Insight 2 (heap cho tập sẵn sàng):** tập sẵn sàng chỉ cần "thêm" và "lấy nhỏ nhất theo `(processingTime, index)`" ⇒ min-heap.

**Quy tắc vòng lặp:**
- **Nếu** heap rỗng **thì** `t = max(t, enqueueTime của tác vụ tại p)` (CPU rảnh, nhảy thời gian).
- Đẩy mọi tác vụ đã đến (`enqueueTime ≤ t`) vào heap.
- Lấy tác vụ nhỏ nhất, ghi vào kết quả, `t += processingTime`.

**Invariant:** đầu mỗi vòng, heap chứa **đúng** các tác vụ đã đến (`enqueueTime ≤ t`) mà chưa được xử lý; `t` là thời điểm CPU vừa rảnh.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp theo thời điểm đến + min-heap `(processingTime, index)`

**Giải thích từng đoạn:**
1. `order = sorted(range(n), key=lambda i: tasks[i][0])`.
2. Vòng lặp tới khi đã xử lý `n` tác vụ.
3. Heap rỗng ⇒ nhảy `t`. Đẩy tác vụ đã đến. Lấy min, cộng thời gian.

In [7]:
def order_heap(tasks):
    n = len(tasks)
    order = sorted(range(n), key=lambda i: tasks[i][0])
    res, h, p, t = [], [], 0, 0
    while len(res) < n:
        if not h:
            t = max(t, tasks[order[p]][0])
        while p < n and tasks[order[p]][0] <= t:
            i = order[p]
            heapq.heappush(h, (tasks[i][1], i)); p += 1
        pt, i = heapq.heappop(h)
        res.append(i); t += pt
    return res

print([order_heap(t) for t, _ in examples])

[[0, 2, 3, 1], [4, 3, 2, 0, 1]]


**Code nộp LeetCode (Cách 2):** xem mục 9. **Độ phức tạp:** sắp `O(n log n)`, mỗi tác vụ push/pop một lần `O(log n)` ⇒ `O(n log n)`; Space `O(n)`.

### 7.2 Cách 3 — Hai heap (không sắp trước)

Thay mảng đã sắp bằng heap `pending` khóa `(enqueueTime, index)`. Mỗi bước: nếu heap sẵn sàng rỗng thì nhảy `t` tới `pending[0][0]`; chuyển mọi phần tử `pending` có `enqueueTime ≤ t` sang heap sẵn sàng.

In [8]:
class SolutionTwoHeaps:
    def getOrder(self, tasks: List[List[int]]) -> List[int]:
        pending = [(e, i) for i, (e, _) in enumerate(tasks)]
        heapq.heapify(pending)
        ready, res, t = [], [], 0
        while pending or ready:
            if not ready:
                t = max(t, pending[0][0])
            while pending and pending[0][0] <= t:
                e, i = heapq.heappop(pending)
                heapq.heappush(ready, (tasks[i][1], i))
            pt, i = heapq.heappop(ready)
            res.append(i); t += pt
        return res

print([SolutionTwoHeaps().getOrder(t) for t, _ in examples])

[[0, 2, 3, 1], [4, 3, 2, 0, 1]]


**Độ phức tạp Cách 3:** `heapify` `O(n)` + mỗi tác vụ 2 lần pop và 1 lần push ⇒ `O(n log n)`. Hằng số lớn hơn Cách 2 một chút; ưu điểm khi tác vụ **đến dần** (dữ liệu luồng) — không cần biết hết trước để sắp.

### 7.3 Cách 4 — Mô phỏng bằng `sorted` + quét con trỏ, chọn min trong một "cửa sổ" danh sách (không heap)

Cải tiến một phần của vét cạn: vẫn sắp theo thời điểm đến, nhưng tập sẵn sàng là một **list** và chọn min bằng `min()` rồi xóa. Thêm vào `O(1)`, chọn `O(|ready|)` ⇒ tệ nhất `O(n²)` (khi mọi tác vụ đến cùng lúc), nhưng nhanh khi tập sẵn sàng luôn nhỏ. Minh họa để thấy rõ vai trò của heap.

In [9]:
def order_list_ready(tasks):
    n = len(tasks); order = sorted(range(n), key=lambda i: tasks[i][0])
    res, ready, p, t = [], [], 0, 0
    while len(res) < n:
        if not ready:
            t = max(t, tasks[order[p]][0])
        while p < n and tasks[order[p]][0] <= t:
            ready.append(order[p]); p += 1
        j = min(ready, key=lambda i: (tasks[i][1], i))
        ready.remove(j); res.append(j); t += tasks[j][1]
    return res

print([order_list_ready(t) for t, _ in examples])

[[0, 2, 3, 1], [4, 3, 2, 0, 1]]


**Độ phức tạp Cách 4:** `O(n log n + Σ|ready|)`, tệ nhất `O(n²)`.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 1**, in `t`, các tác vụ được đẩy vào heap, nội dung heap và lựa chọn:

In [10]:
def trace_heap(tasks):
    n = len(tasks); order = sorted(range(n), key=lambda i: tasks[i][0])
    res, h, p, t = [], [], 0, 0
    while len(res) < n:
        if not h:
            nt = max(t, tasks[order[p]][0])
            if nt != t: print(f"  CPU rảnh: nhảy t {t} -> {nt}")
            t = nt
        pushed = []
        while p < n and tasks[order[p]][0] <= t:
            i = order[p]; heapq.heappush(h, (tasks[i][1], i)); pushed.append(i); p += 1
        pt, i = heapq.heappop(h)
        print(f"t={t:>2}: đẩy {pushed}, heap trước khi lấy = {sorted(h + [(pt, i)])} -> chọn tác vụ {i} (xử lý {pt}), xong lúc {t + pt}")
        res.append(i); t += pt
    return res

assert trace_heap(examples[0][0]) == [0, 2, 3, 1]

  CPU rảnh: nhảy t 0 -> 1
t= 1: đẩy [0], heap trước khi lấy = [(2, 0)] -> chọn tác vụ 0 (xử lý 2), xong lúc 3
t= 3: đẩy [1, 2], heap trước khi lấy = [(2, 2), (4, 1)] -> chọn tác vụ 2 (xử lý 2), xong lúc 5
t= 5: đẩy [3], heap trước khi lấy = [(1, 3), (4, 1)] -> chọn tác vụ 3 (xử lý 1), xong lúc 6
t= 6: đẩy [], heap trước khi lấy = [(4, 1)] -> chọn tác vụ 1 (xử lý 4), xong lúc 10


Trace ví dụ tự dựng có **khoảng rảnh** `[[1,1],[10,2],[10,1]]` (ví dụ tự dựng — đối chiếu brute force; nên đối chiếu lại đề gốc):

In [11]:
tk = [[1, 1], [10, 2], [10, 1]]
r = trace_heap(tk); assert r == order_bruteforce(tk) == [0, 2, 1]

  CPU rảnh: nhảy t 0 -> 1
t= 1: đẩy [0], heap trước khi lấy = [(1, 0)] -> chọn tác vụ 0 (xử lý 1), xong lúc 2
  CPU rảnh: nhảy t 2 -> 10
t=10: đẩy [1, 2], heap trước khi lấy = [(1, 2), (2, 1)] -> chọn tác vụ 2 (xử lý 1), xong lúc 11
t=11: đẩy [], heap trước khi lấy = [(2, 1)] -> chọn tác vụ 1 (xử lý 2), xong lúc 13


| t | sự kiện | heap | chọn |
|---|---|---|---|
| 1 | đẩy 0 | {(1,0)} | 0, xong lúc 2 |
| 2 → 10 | heap rỗng, nhảy t | — | — |
| 10 | đẩy 1, 2 | {(1,2), (2,1)} | 2, xong lúc 11 |
| 11 | — | {(2,1)} | 1 |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — sắp theo thời điểm đến + min-heap.

In [12]:
class Solution:
    def getOrder(self, tasks: List[List[int]]) -> List[int]:
        n = len(tasks)
        order = sorted(range(n), key=lambda i: tasks[i][0])
        res = []
        heap = []
        p = 0
        t = 0
        while len(res) < n:
            if not heap:
                t = max(t, tasks[order[p]][0])
            while p < n and tasks[order[p]][0] <= t:
                i = order[p]
                heapq.heappush(heap, (tasks[i][1], i))
                p += 1
            pt, i = heapq.heappop(heap)
            res.append(i)
            t += pt
        return res

for t, exp in examples:
    got = Solution().getOrder(t); print(got, exp); assert got == exp

[0, 2, 3, 1] [0, 2, 3, 1]
[4, 3, 2, 0, 1] [4, 3, 2, 0, 1]


## 10. Vì sao thuật toán đúng

Ta chứng minh thuật toán chọn **cùng** tác vụ với CPU mô tả trong đề ở mọi bước, bằng quy nạp theo số tác vụ đã xử lý.

**Invariant:** đầu mỗi vòng, `t` = thời điểm CPU rảnh (xong tác vụ trước, hoặc 0 lúc đầu), heap = tập tác vụ có `enqueueTime ≤ t` chưa xử lý, và mọi tác vụ chưa vào heap có `enqueueTime > t` (nhờ thứ tự sắp và con trỏ `p`).

**Một bước:**
- Nếu heap rỗng: theo đề, CPU rảnh cho tới khi có tác vụ đến; tác vụ đến sớm nhất là `order[p]` ⇒ CPU bắt đầu chọn tại `t' = enqueueTime(order[p])` (hoặc `t` nếu lớn hơn). Đặt `t = max(t, …)` khớp điều này.
- Đẩy mọi tác vụ có `enqueueTime ≤ t` ⇒ heap = **đúng** tập sẵn sàng tại thời điểm CPU chọn.
- CPU chọn tác vụ có `(processingTime, index)` nhỏ nhất trong tập sẵn sàng — chính là `heappop`.
- CPU xử lý trọn tác vụ, rảnh lại tại `t + processingTime`; các tác vụ đến trong lúc đó chờ (sẽ được đẩy ở vòng sau). Invariant giữ.

Vậy dãy chọn trùng với hành vi CPU. ∎

## 11. Phân tích Time / Space Complexity

Gọi `n = len(tasks) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng quét toàn bộ | `O(n²)` | `O(n)` |
| 2. Sắp + heap | `O(n log n)` | `O(n)` |
| 3. Hai heap | `O(n log n)` | `O(n)` |
| 4. Sắp + list sẵn sàng | `O(n log n + Σ|ready|)`, tệ nhất `O(n²)` | `O(n)` |

Thời gian `t` có thể lên tới `10⁹ + 10⁵·10⁹ ≈ 10¹⁴` — Python không tràn số; ở C++/Java phải dùng `long`.

In [13]:
rng = random.Random(1834)
same_time = [[1, rng.randint(1, 10 ** 9)] for _ in range(3000)]               # mọi tác vụ đến cùng lúc: tệ nhất cho Cách 4
spread = [[rng.randint(1, 10 ** 9), rng.randint(1, 10 ** 4)] for _ in range(10 ** 5)]
for name, f in [("Cách 2", Solution().getOrder), ("Cách 3", SolutionTwoHeaps().getOrder), ("Cách 4", order_list_ready)]:
    for label, data in [("3000 cùng lúc", same_time), ("1e5 rải rác", spread)]:
        t0 = time.perf_counter(); f(data); print(f"{name} | {label:<13}: {(time.perf_counter()-t0)*1000:7.1f} ms")
t0 = time.perf_counter(); order_bruteforce(same_time[:1500]); print(f"Cách 1 | 1500 cùng lúc : {(time.perf_counter()-t0)*1000:7.1f} ms")

Cách 2 | 3000 cùng lúc:     1.5 ms
Cách 2 | 1e5 rải rác  :    64.9 ms
Cách 3 | 3000 cùng lúc:     1.8 ms


Cách 3 | 1e5 rải rác  :   179.1 ms


Cách 4 | 3000 cùng lúc:   257.2 ms
Cách 4 | 1e5 rải rác  :    89.2 ms


Cách 1 | 1500 cùng lúc :   118.4 ms


## 12. Bộ test

Đối chiếu mọi cách với mô phỏng vét cạn trên tác vụ ngẫu nhiên (seed cố định) với miền giá trị **nhỏ** để có nhiều hòa và khoảng rảnh; kiểm tra thêm: kết quả là một **hoán vị** của `0..n−1`, và input **không bị thay đổi**.

In [14]:
ALL = {"heap": order_heap, "twoHeaps": SolutionTwoHeaps().getOrder, "listReady": order_list_ready, "submit": Solution().getOrder}
def run_all(tasks, with_brute=True):
    snapshot = [list(x) for x in tasks]
    res = {k: f(tasks) for k, f in ALL.items()}
    if with_brute:
        res["brute"] = order_bruteforce(tasks)
    assert tasks == snapshot, "input bị thay đổi!"
    vals = {tuple(v) for v in res.values()}
    assert len(vals) == 1, (tasks, res)
    ans = res["submit"]; assert sorted(ans) == list(range(len(tasks)))
    return ans

for t, exp in examples:
    assert run_all(t) == exp
edge = [([[1, 1]], [0]), ([[5, 2], [1, 3]], [1, 0]), ([[1, 5], [1, 5], [1, 5]], [0, 1, 2]), ([[1, 10], [2, 1], [3, 1]], [0, 1, 2]),
        ([[1, 1], [10, 2], [10, 1]], [0, 2, 1]), ([[10 ** 9, 10 ** 9], [1, 10 ** 9]], [1, 0])]
for t, exp in edge:
    assert run_all(t) == exp, t
rng = random.Random(18340)
for _ in range(500):
    n = rng.randint(1, 12); E = rng.choice([3, 10, 50]); P = rng.choice([2, 5, 20])
    run_all([[rng.randint(1, E), rng.randint(1, P)] for _ in range(n)])
for _ in range(10):
    run_all([[rng.randint(1, 10 ** 6), rng.randint(1, 10 ** 3)] for _ in range(3000)], with_brute=False)
print("Ví dụ, edge case, 500 test có vét cạn, 10 test 3000 tác vụ: OK")

Ví dụ, edge case, 500 test có vét cạn, 10 test 3000 tác vụ: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên nhảy thời gian khi CPU rảnh (heap rỗng)

In [15]:
def wrong_no_jump(tasks):
    n = len(tasks); order = sorted(range(n), key=lambda i: tasks[i][0])
    res, h, p, t = [], [], 0, 0
    guard = 0
    while len(res) < n:
        guard += 1
        if guard > 10 * n + 10:
            return f"KẸT sau {guard} vòng (res={res})"
        while p < n and tasks[order[p]][0] <= t:
            i = order[p]; heapq.heappush(h, (tasks[i][1], i)); p += 1
        if not h:
            continue                                      # SAI: t không đổi -> lặp mãi
        pt, i = heapq.heappop(h); res.append(i); t += pt
    return res

print("sai :", wrong_no_jump([[5, 1], [1, 1]]), "| đúng:", Solution().getOrder([[5, 1], [1, 1]]))

sai : KẸT sau 31 vòng (res=[]) | đúng: [1, 0]


**Vì sao sai:** khi không có tác vụ sẵn sàng, `t` không bao giờ tăng ⇒ vòng lặp vô hạn (ta cài "cầu chì" để dừng). **Sửa:** `if not heap: t = max(t, enqueueTime kế tiếp)`.

### Lỗi 2 — Đưa `enqueueTime` vào khóa heap

In [16]:
def wrong_key(tasks):
    n = len(tasks); order = sorted(range(n), key=lambda i: tasks[i][0])
    res, h, p, t = [], [], 0, 0
    while len(res) < n:
        if not h: t = max(t, tasks[order[p]][0])
        while p < n and tasks[order[p]][0] <= t:
            i = order[p]; heapq.heappush(h, (tasks[i][0], tasks[i][1], i)); p += 1     # SAI
        _, pt, i = heapq.heappop(h); res.append(i); t += pt
    return res

t, exp = examples[0]
print("sai :", wrong_key(t), "| đúng:", exp)

sai : [0, 1, 2, 3] | đúng: [0, 2, 3, 1]


**Vì sao sai:** khi CPU chọn, mọi tác vụ trong heap đều đã sẵn sàng — thời điểm đến **không** còn ý nghĩa; khóa phải là `(processingTime, index)`. Ở đây code sai chọn tác vụ 1 (đến lúc 2) trước tác vụ 2 (ngắn hơn). **Sửa:** khóa `(processingTime, index)`.

### Lỗi 3 — Hòa thời gian xử lý mà không phá hòa theo chỉ số (dùng `(processingTime, enqueueTime)`)

In [17]:
def wrong_tie(tasks):
    n = len(tasks); order = sorted(range(n), key=lambda i: tasks[i][0])
    res, h, p, t = [], [], 0, 0
    while len(res) < n:
        if not h: t = max(t, tasks[order[p]][0])
        while p < n and tasks[order[p]][0] <= t:
            i = order[p]; heapq.heappush(h, (tasks[i][1], -tasks[i][0], i)); p += 1   # SAI: hòa thì ưu tiên đến muộn
        pt, _, i = heapq.heappop(h); res.append(i); t += pt
    return res

tk = [[1, 5], [2, 2], [3, 2]]                       # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_tie(tk), "| đúng:", Solution().getOrder(tk), "| brute:", order_bruteforce(tk))

sai : [0, 2, 1] | đúng: [0, 1, 2] | brute: [0, 1, 2]


**Vì sao sai:** lúc `t = 6`, tác vụ 1 và 2 cùng thời gian xử lý 2; đề phá hòa bằng **chỉ số nhỏ nhất** (chọn 1), còn code sai ưu tiên tác vụ đến muộn hơn (chọn 2). **Sửa:** phần tử thứ hai của khóa là `index`.

### Lỗi 4 — Chỉ đẩy **một** tác vụ mỗi vòng (thay vì mọi tác vụ đã đến)

In [18]:
def wrong_push_one(tasks):
    n = len(tasks); order = sorted(range(n), key=lambda i: tasks[i][0])
    res, h, p, t = [], [], 0, 0
    while len(res) < n:
        if not h: t = max(t, tasks[order[p]][0])
        if p < n and tasks[order[p]][0] <= t:            # SAI: if thay vì while
            i = order[p]; heapq.heappush(h, (tasks[i][1], i)); p += 1
        pt, i = heapq.heappop(h); res.append(i); t += pt
    return res

t, exp = examples[1]
print("sai :", wrong_push_one(t), "| đúng:", exp)

sai : [0, 1, 2, 3, 4] | đúng: [4, 3, 2, 0, 1]


**Vì sao sai:** ở Ví dụ 2 cả 5 tác vụ đến lúc 7, nhưng mỗi vòng chỉ đẩy 1 ⇒ CPU chọn trong tập thiếu ⇒ thứ tự sai. **Sửa:** `while` đẩy **hết** các tác vụ có `enqueueTime ≤ t`.

### Lỗi 5 — Sắp `tasks` tại chỗ rồi trả chỉ số **sau khi sắp**

In [19]:
def wrong_sorted_index(tasks):
    ts = sorted(tasks)                                   # SAI: mất chỉ số gốc
    res, h, p, t = [], [], 0, 0; n = len(ts)
    while len(res) < n:
        if not h: t = max(t, ts[p][0])
        while p < n and ts[p][0] <= t:
            heapq.heappush(h, (ts[p][1], p)); p += 1
        pt, i = heapq.heappop(h); res.append(i); t += pt
    return res

t, exp = examples[1]
print("sai :", wrong_sorted_index(t), "| đúng:", exp)

sai : [0, 1, 2, 3, 4] | đúng: [4, 3, 2, 0, 1]


**Vì sao sai:** sau khi sắp, vị trí trong mảng không còn là chỉ số gốc; còn phá hòa theo "chỉ số sau sắp" cũng sai. **Sửa:** sắp **chỉ số** (`order = sorted(range(n), key=…)`) hoặc gắn chỉ số gốc vào mỗi tác vụ trước khi sắp.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mô phỏng hàng đợi công việc theo thời gian, chọn theo ưu tiên" ⇒ **sắp theo thời điểm đến + heap theo ưu tiên**.
- Khóa heap = **đúng** các tiêu chí chọn của đề, theo thứ tự (ở đây `(processingTime, index)`).
- Luôn xử lý trường hợp **rảnh** bằng nhảy thời gian, không tăng `t` từng đơn vị.
- Đẩy **mọi** sự kiện đã tới (`while`, không `if`).
- Giữ chỉ số gốc khi sắp.

## 15. Biến thể của bài

1. **Nhiều CPU (k luồng)**: thêm heap thời điểm rảnh của các CPU — **1882 Process Tasks Using Servers**.
2. **Cho phép ngắt (preemptive SJF / SRTF)**: khi có tác vụ mới ngắn hơn phần còn lại thì đổi — cần xử lý sự kiện "đến" cả trong lúc đang chạy.
3. **Tính thời gian chờ trung bình** (**1701 Average Waiting Time** — FIFO, không heap).
4. **630 Course Schedule III**, **621 Task Scheduler**: heap + tham lam theo thời gian.
5. **Trả về thời điểm bắt đầu/kết thúc của từng tác vụ** (Bài tập 4).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét toàn bộ | Đúng mô tả, mỗi bước lọc + min | `O(n²)` | `O(n)` | Kiểm chứng |
| 2. Sắp + heap | Con trỏ đưa tác vụ vào heap | `O(n log n)` | `O(n)` | **Bản nộp** |
| 3. Hai heap | Heap đến + heap sẵn sàng | `O(n log n)` | `O(n)` | Tác vụ đến dạng luồng |
| 4. Sắp + list | min trên list sẵn sàng | tệ nhất `O(n²)` | `O(n)` | Minh họa vì sao cần heap |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[[1,3],[1,2],[1,1]]`; (b) `[[1,1],[2,1],[3,1]]`; (c) `[[3,1],[1,5]]`; (d) `[[1,4],[2,1],[2,1],[6,1]]`.

**Bài 2 (tính tay):** Lập bảng `t`, heap, lựa chọn cho `tasks = [[2,3],[1,4],[4,1],[4,2]]`.

**Bài 3 (điền TODO):** Hoàn thiện `push_arrived(tasks, order, p, t, heap)` đẩy mọi tác vụ đã đến và trả `p` mới.

**Bài 4 (tự viết lại):** Viết `schedule(tasks)` trả list `(index, start, end)` theo thứ tự xử lý; kiểm tra: `start ≥ enqueueTime`, các đoạn không chồng nhau.

**Bài 5 (mini challenge):** Viết `get_order_k(tasks, k)` cho **k CPU** giống hệt nhau (mỗi CPU rảnh thì chọn tác vụ như đề; nhiều CPU rảnh cùng lúc thì CPU chỉ số nhỏ chọn trước). Trả list `(index, cpu)`. Đối chiếu một mô phỏng vét cạn từng bước.

In [20]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def push_arrived_todo(tasks, order, p, t, heap):
    # TODO: while p < len(order) và tasks[order[p]][0] <= t: đẩy (processingTime, index), p += 1
    return p

In [21]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def schedule(tasks):
    # TODO: như Cách 2, lưu (i, t, t + pt) trước khi cộng t
    pass

def get_order_k(tasks, k):
    # TODO: heap CPU rảnh (thời điểm rảnh, chỉ số cpu); heap sẵn sàng (pt, i); con trỏ theo thời điểm đến
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tác vụ nào đến trước. Mức 2: tập sẵn sàng ở mỗi lần chọn. Mức 3: (a) `[2, 1, 0]`; (b) `[0, 1, 2]`; (c) `[1, 0]`; (d) `[0, 1, 2, 3]`: làm 0 tới lúc 5; lúc 5 sẵn sàng {1, 2} cùng thời gian 1 ⇒ chọn 1; lúc 6 sẵn sàng {2, 3} ⇒ chọn 2 (chỉ số nhỏ), rồi 3.

**Bài 2.** Mức 1: `t = 1` chỉ có tác vụ 1 (4) ⇒ làm 1, xong 5. Mức 2: `t = 5`: {0 (3), 2 (1), 3 (2)} ⇒ 2, xong 6. Mức 3: `t = 6`: {0 (3), 3 (2)} ⇒ 3, rồi 0 ⇒ `[1, 2, 3, 0]`.

**Bài 3.** Mức 1: `while`. Mức 2: `heappush(heap, (tasks[i][1], i))`. Mức 3: trả `p`.

**Bài 4.** Mức 1: trước `t += pt` ghi `(i, t, t + pt)`. Mức 2: kiểm `start ≥ tasks[i][0]`. Mức 3: `end` của tác vụ trước `≤ start` của tác vụ sau.

**Bài 5.** Mức 1: ba heap — CPU rảnh (theo chỉ số), CPU bận (theo thời điểm rảnh), tác vụ sẵn sàng `(pt, i)`. Mức 2: tại thời điểm `t`: chuyển CPU đã xong sang "rảnh", đẩy tác vụ đã đến, rồi ghép **chừng nào còn cả CPU rảnh lẫn tác vụ sẵn sàng**. Mức 3: thời điểm kế tiếp là sự kiện sớm nhất trong {một CPU xong, một tác vụ đến}; vét cạn mô phỏng đúng luật đó bằng cách quét toàn bộ.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2, 3
b1 = [Solution().getOrder(t) for t in [[[1, 3], [1, 2], [1, 1]], [[1, 1], [2, 1], [3, 1]], [[3, 1], [1, 5]], [[1, 4], [2, 1], [2, 1], [6, 1]]]]
print("Bài 1:", b1); assert b1 == [[2, 1, 0], [0, 1, 2], [1, 0], [0, 1, 2, 3]]
print("Bài 2:"); assert trace_heap([[2, 3], [1, 4], [4, 1], [4, 2]]) == [1, 2, 3, 0]
def push_arrived_todo(tasks, order, p, t, heap):
    while p < len(order) and tasks[order[p]][0] <= t:
        i = order[p]; heapq.heappush(heap, (tasks[i][1], i)); p += 1
    return p
hp = []; tk = examples[0][0]; od = sorted(range(4), key=lambda i: tk[i][0])
assert push_arrived_todo(tk, od, 0, 3, hp) == 3 and sorted(hp) == [(2, 0), (2, 2), (4, 1)]
print("Bài 3: OK")

Bài 1: [[2, 1, 0], [0, 1, 2], [1, 0], [0, 1, 2, 3]]
Bài 2:
  CPU rảnh: nhảy t 0 -> 1
t= 1: đẩy [1], heap trước khi lấy = [(4, 1)] -> chọn tác vụ 1 (xử lý 4), xong lúc 5
t= 5: đẩy [0, 2, 3], heap trước khi lấy = [(1, 2), (2, 3), (3, 0)] -> chọn tác vụ 2 (xử lý 1), xong lúc 6
t= 6: đẩy [], heap trước khi lấy = [(2, 3), (3, 0)] -> chọn tác vụ 3 (xử lý 2), xong lúc 8
t= 8: đẩy [], heap trước khi lấy = [(3, 0)] -> chọn tác vụ 0 (xử lý 3), xong lúc 11
Bài 3: OK


In [23]:
# Bài 4
def schedule(tasks):
    n = len(tasks); order = sorted(range(n), key=lambda i: tasks[i][0])
    res, h, p, t = [], [], 0, 0
    while len(res) < n:
        if not h: t = max(t, tasks[order[p]][0])
        p = push_arrived_todo(tasks, order, p, t, h)
        pt, i = heapq.heappop(h); res.append((i, t, t + pt)); t += pt
    return res

rng = random.Random(4)
for _ in range(300):
    tk = [[rng.randint(1, 10), rng.randint(1, 5)] for _ in range(rng.randint(1, 10))]
    sc = schedule(tk)
    assert [i for i, _, _ in sc] == Solution().getOrder(tk)
    assert all(s >= tk[i][0] and e - s == tk[i][1] for i, s, e in sc)
    assert all(sc[j][2] <= sc[j + 1][1] for j in range(len(sc) - 1))
print("Bài 4: OK |", schedule(examples[0][0]))

Bài 4: OK | [(0, 1, 3), (2, 3, 5), (3, 5, 6), (1, 6, 10)]


In [24]:
# Bài 5
def get_order_k(tasks, k):
    n = len(tasks); order = sorted(range(n), key=lambda i: tasks[i][0])
    idle = list(range(k)); heapq.heapify(idle)           # CPU rảnh, ưu tiên chỉ số nhỏ
    busy = []                                            # (thời điểm rảnh, cpu)
    ready, res, p, t = [], [], 0, 0
    while len(res) < n:
        while busy and busy[0][0] <= t:
            heapq.heappush(idle, heapq.heappop(busy)[1])
        p = push_arrived_todo(tasks, order, p, t, ready)
        while idle and ready:
            pt, i = heapq.heappop(ready); c = heapq.heappop(idle)
            res.append((i, c)); heapq.heappush(busy, (t + pt, c))
        nxt = []
        if busy: nxt.append(busy[0][0])
        if p < n: nxt.append(tasks[order[p]][0])
        if len(res) < n:
            t = max(t, min(nxt)) if idle and not ready else min(nxt)
    return res

def k_brute(tasks, k):
    n = len(tasks); done = [False] * n; free = [0] * k; res = []; t = 0
    while len(res) < n:
        assigned = True
        while assigned:
            assigned = False
            idle = [c for c in range(k) if free[c] <= t]
            avail = [i for i in range(n) if not done[i] and tasks[i][0] <= t]
            if idle and avail:
                j = min(avail, key=lambda i: (tasks[i][1], i)); c = min(idle)
                done[j] = True; res.append((j, c)); free[c] = t + tasks[j][1]; assigned = True
        events = [f for f in free if f > t] + [tasks[i][0] for i in range(n) if not done[i] and tasks[i][0] > t]
        if events: t = min(events)
    return res

rng = random.Random(5)
for _ in range(300):
    tk = [[rng.randint(1, 8), rng.randint(1, 4)] for _ in range(rng.randint(1, 9))]
    assert [i for i, _ in get_order_k(tk, 1)] == Solution().getOrder(tk)
    kk = rng.randint(1, 3)
    assert get_order_k(tk, kk) == k_brute(tk, kk), (tk, kk)
print("Bài 5: OK |", get_order_k([[1, 2], [2, 4], [3, 2], [4, 1]], 2))

Bài 5: OK | [(0, 0), (1, 1), (2, 0), (3, 0)]


Ghi chú Bài 5: luật "nhiều CPU rảnh cùng lúc thì CPU chỉ số nhỏ chọn trước" là **luật tự đặt** cho biến thể (không có trong đề gốc; nên đối chiếu lại nếu gặp bài thật như 1882).

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "sắp theo thời điểm đến + heap ưu tiên":**

```python
order = sorted(range(n), key=lambda i: tasks[i][0])
res, heap, p, t = [], [], 0, 0
while len(res) < n:
    if not heap: t = max(t, tasks[order[p]][0])          # CPU rảnh: nhảy thời gian
    while p < n and tasks[order[p]][0] <= t:              # đẩy MỌI tác vụ đã đến
        i = order[p]; heappush(heap, (tasks[i][1], i)); p += 1
    pt, i = heappop(heap); res.append(i); t += pt
return res
```

**Checklist:**
- [ ] Khóa heap là `(processingTime, index)` (không chứa `enqueueTime`)?
- [ ] Nhảy `t` khi heap rỗng?
- [ ] `while` đẩy mọi tác vụ đã đến?
- [ ] Giữ **chỉ số gốc** khi sắp?
- [ ] Đã test: mọi tác vụ đến cùng lúc, có khoảng rảnh, hòa thời gian xử lý, giá trị `10⁹`?